# Fase 4: Pengujian Hipotesis (Z-Test)
**Member:** Safani Nuraini — Hypothesis Analyst

**Pertanyaan Riset (P2):** Apakah terdapat perbedaan waktu penyelesaian (*time-to-close*) yang signifikan secara statistik antara *issue/PR* yang ditangani oleh Core Member dengan Contributor eksternal?

**Tujuan:** Menerapkan 6 langkah baku uji hipotesis menggunakan uji-Z dua sampel independen untuk membandingkan rata-rata waktu penyelesaian (bukan proporsi), serta memberikan interpretasi statistik yang valid tanpa melanggar batasan terminologi.

## AI Usage Disclosure
**Member:** Safani Nuraini — Hypothesis Analyst | **Tools used:** None

| Task | Tool | Prompt summary | Output modified? |
| --- | --- | --- | --- |
| - | None | - | - |

**Written entirely without AI:** Seluruh penulisan rumusan statistik (H0 dan H1), penetapan tingkat signifikansi, analisis *critical region*, evaluasi matematis *p-value*, perumusan keputusan akhir ("reject H0" / "fail to reject H0"), serta seluruh narasi interpretasi konteks proyek dikerjakan 100% manual secara mandiri tanpa bantuan AI.

In [1]:
import sys
import pandas as pd
import numpy as np
import scipy.stats as stats

sys.path.append("..")
# Import fungsi beda rata-rata (bukan proporsi) milik Member D
from src.hypotesis import z_test_two_sample

# 1. Muat data bersih yang sudah disiapkan oleh Member A
df = pd.read_csv("../data/clean/dataset.csv")

# 2. Konversi format waktu dan hitung time-to-close (dalam satuan jam)
df['created_at'] = pd.to_datetime(df['created_at'])
df['closed_at'] = pd.to_datetime(df['closed_at'])
df['time_to_close_hours'] = (df['closed_at'] - df['created_at']).dt.total_seconds() / 3600

# Drop baris data yang belum closed/merged (menghindari NaN)
df_closed = df.dropna(subset=['time_to_close_hours']).copy()

### Langkah 1: Menentukan Hipotesis Nol (H0) dan Hipotesis Alternatif (Ha)
Karena kita menganalisis waktu (variabel kontinu), parameter yang diuji adalah Rata-Rata ($\mu$), BUKAN proporsi ($p$).
* **H0:** $\mu_{core} = \mu_{contributor}$ (Tidak ada perbedaan rata-rata waktu penyelesaian *issue* antara *Core Member* dan *Contributor*).
* **Ha:** $\mu_{core} \neq \mu_{contributor}$ (Terdapat perbedaan rata-rata waktu penyelesaian *issue* antara *Core Member* dan *Contributor*).

### Langkah 2: Tingkat Signifikansi ($\alpha$)
* **$\alpha$ = 0.05** (Menggunakan tingkat signifikansi standar 5%).

### Langkah 3: Statistik Uji
Karena parameter populasi tidak diketahui namun ukuran sampel ($n$) jauh lebih besar dari 30, kita secara sah dapat menggunakan **Z-Test Dua Sampel Independen** (perbandingan dua rata-rata) berdasarkan *Central Limit Theorem*.

### Langkah 4: Critical Region (Daerah Penolakan)
Dengan pengujian dua arah (*two-tailed*) pada $\alpha = 0.05$, nilai kritis Z adalah **$\pm 1.96$**. 
* **Aturan Keputusan:** H0 akan ditolak jika nilai $|Z_{stat}| > 1.96$ atau jika $p\text{-value} < 0.05$.

In [4]:
core_data = df_closed[df_closed['author_association'] == 'MEMBER']['time_to_close_hours'].values
contrib_data = df_closed[df_closed['author_association'] == 'CONTRIBUTOR']['time_to_close_hours'].values

# Kalkulasi parameter deskriptif masing-masing sampel (n, rata-rata, standar deviasi sampel)
n1 = len(core_data)
x_bar1 = np.mean(core_data)
sigma1 = np.std(core_data, ddof=1) 

n2 = len(contrib_data)
x_bar2 = np.mean(contrib_data)
sigma2 = np.std(contrib_data, ddof=1)

print(f"Statistik Core Member : n={n1}, rata-rata={x_bar1:.2f} jam, std={sigma1:.2f}")
print(f"Statistik Contributor : n={n2}, rata-rata={x_bar2:.2f} jam, std={sigma2:.2f}")

Statistik Core Member : n=742, rata-rata=120.65 jam, std=255.10
Statistik Contributor : n=614, rata-rata=220.16 jam, std=454.22


In [5]:
# Langkah 5: Hitung nilai Z dan P-Value memanggil fungsi di hypothesis.py
test_result = z_test_two_sample(
    x_bar1=x_bar1, x_bar2=x_bar2, 
    sigma1=sigma1, sigma2=sigma2, 
    n1=n1, n2=n2, 
    alternative='two-sided', alpha=0.05
)

print(f"Z-Statistic : {test_result['z_stat']:.4f}")
print(f"P-Value     : {test_result['p_value']:.4e}")
print(f"Keputusan   : {test_result['decision']}")

Z-Statistic : -4.8341
P-Value     : 1.3376e-06
Keputusan   : reject H0


### Langkah 6: Keputusan & Interpretasi

**Keputusan Statistik:**
Berdasarkan hasil uji-Z Dua Sampel di atas, perbandingan nilai $p\text{-value}$ (`1.3376e-06`) terhadap tingkat signifikansi $\alpha = 0.05$ menghasilkan keputusan **reject H0**.

**Interpretasi Konteks Proyek (pandas-dev/pandas):**
Terdapat bukti statistik yang cukup kuat untuk menyimpulkan bahwa ada perbedaan yang nyata pada rata-rata waktu penyelesaian (*time-to-close*) *issue* antara *Core Member* dan *Contributor* eksternal. *Core Member* terbukti secara rata-rata lebih cepat menyelesaikan *issue* (120.65 jam) dibandingkan *Contributor* (220.16 jam). Perbedaan waktu ini tidak terjadi karena variansi acak, melainkan merepresentasikan karakteristik operasional dan hak akses yang berbeda dalam repositori.

### Kesimpulan Handoff
Tahap Pengujian Hipotesis (*Hypothesis Testing*) menggunakan pendekatan inferensial beda dua rata-rata telah selesai dieksekusi. Pertanyaan Riset 2 (P2) telah terjawab secara sah melalui uji statistik yang tepat (tanpa asumsi proporsi yang keliru). 

Distribusi waktu penyelesaian (*time-to-close*) dan variansi data ini selanjutnya secara resmi diserahkan kepada **Fernando (Member E)** sebagai basis metrik empiris untuk menjalankan Simulasi Komputasi (Monte Carlo & MCMC) guna memprediksi probabilitas *bottleneck* (P3).